<a href="https://colab.research.google.com/github/Numanur/llm-practice/blob/main/LLM_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
from google.colab import drive
drive.mount(
    '/content/drive'
)

Mounted at /content/drive


In [ ]:
!pip install transformers==4.41.2 accelerate==0.30.1

In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=True,
    attn_implementation="eager"   # 👈 important fix
)

tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    trust_remote_code=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    return_full_text = False,
    max_new_tokens = 500,
    do_sample = False
)

In [ ]:
messages = [
    {"role": "user", "content":"Create a funny joke about Chatgpt."}
]
output = generator(messages)
print(output[0]["generated_text"])

 Why don't you ever play hide and seek with Chatgpt? Because good luck hiding when it's always ready to find you!


# **Dense Retrieval for RAG**

In [3]:
!pip install cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.5/350.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 93.2 MB/s eta 0:00:00


In [4]:
import cohere
import numpy as np
import pandas as pd
from tqdm import tqdm

api_key = 'BxCtQ0QTet1dFvpLfT3rfk7pPNPsiIHATZheTjpb'
co =  cohere.Client(api_key)

In [5]:
text = """
Interstellar is a 2014 epic science fiction film co-written,
directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain,
Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to
survive, the film follows a group of astronauts who travel
through a wormhole near Saturn in search of a new home for
mankind.
Brothers Christopher and Jonathan Nolan wrote the screenplay,
which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in
Physics[4] Kip Thorne was an executive producer, acted as a
scientific consultant, and wrote a tie-in book, The Science of
Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in
the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in
Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and
the company Double Negative created additional digital effects.
Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock,
expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773
million with subsequent re-releases), making it the tenth-highest
grossing film of 2014.
It received acclaim for its performances, direction, screenplay,
musical score, visual effects, ambition, themes, and emotional
weight.
It has also received praise from many astronomers for its
scientific accuracy and portrayal of theoretical astrophysics.
Since its premiere, Interstellar gained a cult following,[5] and
now is regarded by many sci-fi experts as one of the best
science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy
Awards, winning Best Visual Effects, and received numerous other
accolades"""


In [6]:
texts = text.split(".")
texts = [t.strip(' \n') for t in texts]


In [9]:
response = co.embed(
    texts=texts,
    model="embed-v4.0",   # or "embed-v4.0"
    input_type="search_document",
).embeddings

embeds = np.array(response)
print(embeds.shape)

(15, 1536)


In [18]:
!pip install -q sentence-transformers faiss-cpu

In [19]:
from sentence_transformers import SentenceTransformer

In [12]:
import faiss
dim = embeds.shape[1]
index = faiss.IndexFlatL2(dim)
print(index.is_trained)
index.add(np.float32(embeds))


True


In [15]:
def search(query, number_of_results=3):
    query_embed = co.embed(
        texts=[query],
        model="embed-v4.0",
        input_type="search_query",
    ).embeddings[0]

    distances, similar_item_ids = index.search(
        np.float32([query_embed]),
        number_of_results
    )

    texts_np = np.array(texts)
    results = pd.DataFrame({
        "texts": texts_np[similar_item_ids[0]],
        "distance": distances[0]
    })

    print(f"Query: {query}\nNearest neighbors:")
    return results

In [17]:
query = "how far the moon is"
result = search(query)
result

Query: how far the moon is
Nearest neighbors:


,texts,distance
0,The film had a worldwide gross over $677 milli...,1.617090
1,It has also received praise from many astronom...,1.633544
2,Set in a dystopian future where humanity is st...,1.658736


# **Loading local models instead of Cohere**

In [24]:
# 1) Load local embedding model
model = SentenceTransformer("BAAI/bge-base-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [25]:
embeds = model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
print(embeds.shape)

(15, 768)


In [26]:
dim = embeds.shape[1]
index = faiss.IndexFlatIP(dim)   # use IP with normalized embeddings for cosine similarity
index.add(np.float32(embeds))

In [27]:
# 5) Search function
def search(query, number_of_results=3):
    query_embed = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, similar_item_ids = index.search(np.float32(query_embed), number_of_results)

    texts_np = np.array(texts)
    results = pd.DataFrame({
        "texts": texts_np[similar_item_ids[0]],
        "score": scores[0]
    })

    print(f"Query: {query}\nNearest neighbors:")
    return results

In [29]:
# 6) Run search
query = "how precise was the science"
result = search(query)
result

Query: how precise was the science
Nearest neighbors:


,texts,score
0,It has also received praise from many astronom...,0.614431
1,Caltech theoretical physicist and 2017 Nobel l...,0.518071
2,"Since its premiere, Interstellar gained a cult...",0.473704


In [32]:
import nbformat

# Load your notebook file
file_path = '/content/drive/MyDrive/Colab Notebooks/LLM_1.ipynb'
with open(file_path, 'r', encoding='utf-8') as f:
    nb = nbformat.read(f, as_version=4)

# Remove the problematic widgets metadata
if 'widgets' in nb.metadata:
    del nb.metadata['widgets']
    print("Metadata 'widgets' removed successfully.")

# Save the clean version
with open('clean_notebook.ipynb', 'w', encoding='utf-8') as f:
    nbformat.write(nb, f)

Metadata 'widgets' removed successfully.
